In [ ]:
# %% [markdown]
# 
# # Example of High Gamma Filter
# 
# Below is a code sample for extracting high gamma power from a raw data file, followed by permutation cluster stats on that high gamma power data
# 

# %% [markdown]
# ### working version 12/1/23

# %% [markdown]

# %% [markdown]
# use window stats with perm testing (0 to 0.5, 0.5 to 1, 0 to 1 sec relative to stim onset)

# %%
import sys
import os
print(sys.path)
# sys.path.append("C:/Users/jz421/Desktop/GlobalLocal/IEEG_Pipelines/") #need to do this cuz otherwise ieeg isn't added to path...comment out when running on cluster, but uncomment when running on pc.

# Get the absolute path to the directory containing the current script
# For GlobalLocal/src/analysis/preproc/make_epoched_data.py, this is GlobalLocal/src/analysis/preproc
current_script_dir = os.path.dirname(os.path.abspath(__file__))

# Navigate up three levels to get to the 'GlobalLocal' directory
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..', '..'))

# Add the 'GlobalLocal' directory to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.insert(0, project_root) # insert at the beginning to prioritize it

import pandas as pd
import json
from statsmodels.stats.multitest import multipletests
from ieeg.navigate import channel_outlier_marker, trial_ieeg, crop_empty_data, \
    outliers_to_nan
from ieeg.io import raw_from_layout, get_data
from ieeg.timefreq.utils import crop_pad
from ieeg.timefreq import gamma
from ieeg.calc.scaling import rescale
import mne
import numpy as np
from ieeg.calc.stats import time_perm_cluster
from ieeg.calc.fast import mean_diff
from ieeg.viz.mri import gen_labels
import matplotlib.pyplot as plt
from mne.utils import fill_doc, verbose
import random
from contextlib import redirect_stdout

print(sys.path)
import pickle
from scipy.stats import ttest_ind
from functools import partial
from src.analysis.utils.general_utils import calculate_RTs, save_channels_to_file, save_sig_chans, load_sig_chans, identify_bad_channels_by_trial_nan_rate, impute_trial_nans_by_channel_mean, get_default_LAB_root
from src.analysis.utils.epoch_metadata_utils import make_metadata_from_event_names, add_previous_trial_info


SyntaxError: trailing comma not allowed without surrounding parentheses (3542693716.py, line 56)

In [ ]:
# Add this fixed version of crop_empty_data at the top of your script, after the imports
def crop_empty_data_fixed(raw, bound='boundary', start_pad="10s", end_pad="10s"):
    """Fixed version of crop_empty_data that handles MNE compatibility issues."""
    from ieeg.timefreq.utils import to_samples
    
    crop_list = []
    start_pad = to_samples(start_pad, raw.info['sfreq']) / raw.info['sfreq']
    end_pad = to_samples(end_pad, raw.info['sfreq']) / raw.info['sfreq']
    
    # split annotations into blocks
    annot = raw.annotations.copy()
    block_idx = [idx + 1 for idx, val in
                 enumerate(annot) if bound in val['description']]
    block_annot = [annot[i: j] for i, j in
                   zip([0] + block_idx, block_idx +
                       ([len(annot)] if block_idx[-1] != len(annot) else []))]
    
    for block_an in block_annot:
        # remove boundary events from annotations
        no_bound = None
        for an in block_an:
            if bound not in an['description']:
                # FIX: Handle the extras field compatibility issue
                an_dict = {}
                an_dict['onset'] = an['onset']
                an_dict['duration'] = an['duration']
                an_dict['description'] = an['description']
                if 'ch_names' in an:
                    an_dict['ch_names'] = an['ch_names']
                # Don't include 'extras' field to avoid compatibility issues
                
                if no_bound is None:
                    no_bound = mne.Annotations(**an_dict)
                else:
                    # Don't include orig_time in append
                    no_bound.append(onset=an_dict['onset'],
                                  duration=an_dict['duration'],
                                  description=an_dict['description'])
        
        # Skip if block is all boundary events
        if no_bound is None:
            continue
        
        # get start and stop time from raw.annotations onset attribute
        t_min = max(0, no_bound.onset[0] - start_pad)
        t_max = no_bound.onset[-1] + end_pad
        
        # create new cropped raw file
        crop_list.append(raw.copy().crop(tmin=t_min, tmax=t_max))
    
    return mne.concatenate_raws(crop_list)

In [ ]:
# Determine LAB_root based on the operating system and environment
if LAB_root is None:
    HOME = os.path.expanduser("~")
    USER = os.path.basename(HOME)
    
    if os.name == 'nt':  # Windows
        LAB_root = os.path.join(HOME, "Box", "CoganLab")
    elif sys.platform == 'darwin':  # macOS
        LAB_root = os.path.join(HOME, "Library", "CloudStorage", "Box-Box", "CoganLab")
    else:  # Linux (cluster)
        # Check if we're on the cluster by looking for /cwork directory
        if os.path.exists(f"/cwork/{USER}"):
            LAB_root = f"/cwork/{USER}"
        else:
            # Fallback for other Linux systems
            LAB_root = os.path.join(HOME, "CoganLab")
else:
    LAB_root = LAB_root

sub = ['D0057']

layout = get_data(task, root=LAB_root)
filt = raw_from_layout(layout.derivatives['derivatives/clean'], subject=sub,
                    extension='.edf', desc='clean', preload=False)
save_dir = os.path.join(layout.root, 'derivatives', 'freqFilt', 'figs', sub)
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

In [ ]:
print("Use my new crop empty data function that gets rid of the mne annotations extras dict")
good = crop_empty_data_fixed(filt)
# %%

print(f"good channels before dropping bads: {len(good.ch_names)}")
print(f"filt channels before dropping bads: {len(filt.ch_names)}")
all_channels_before_marker = good.ch_names.copy()
channels_to_drop = []

good.info['bads'] = channel_outlier_marker(good, 3, 2)
marker_bad_channels = good.info['bads'].copy()

print("Bad channels in 'good':", good.info['bads'])

filt.drop_channels(marker_bad_channels)  # this has to come first cuz if you drop from good first, then good.info['bads'] is just empty
good.drop_channels(marker_bad_channels)

print("Bad channels in 'good' after dropping once:", good.info['bads'])

print(f"good channels after dropping bads: {len(good.ch_names)}")
print(f"filt channels after dropping bads: {len(filt.ch_names)}")

good.load_data()

# If channels is None, use all channels
if channels is None:
    channels = good.ch_names
else:
    # Validate the provided channels
    invalid_channels = [ch for ch in channels if ch not in good.ch_names]
    if invalid_channels:
        raise ValueError(
            f"The following channels are not valid: {invalid_channels}")

    # Use only the specified channels
    good.pick_channels(channels)

ch_type = filt.get_channel_types(only_data_chs=True)[0]
good.set_eeg_reference(ref_channels="average", ch_type=ch_type)

In [ ]:
trials = trial_ieeg(good, event, times_adj, preload=True, reject_by_annotation=False)

In [ ]:

# within_times_duration = abs(within_base_times[1] - within_base_times[0]) #grab the duration as a string for naming

# <<< Step 1: IDENTIFY bad channels before processing >>>

if outlier_policy == 'drop' or outlier_policy == 'drop_and_nan' or outlier_policy == 'drop_and_impute':
    print(f"--- Identifying Channels with more than {threshold_percent}% outlier trials in the stimulus period (discuss this approach with Greg) ---")
    # Identify channels with too many outlier trials based on the stimulus period
    times_adj = [times[0] - pad_length, times[1] + pad_length]
    event_trials_qc = trial_ieeg(good, "Stimulus", times_adj, preload=True, reject_by_annotation=False)
    outliers_to_nan(event_trials_qc, outliers=outliers)
    channels_to_drop = identify_bad_channels_by_trial_nan_rate(event_trials_qc, threshold_percent)
    print(f"Found {len(channels_to_drop)} bad Stimulus channels: {channels_to_drop}")

    qc_summary = {
        "subject": sub,
        "n_total_channels_before_marker": len(all_channels_before_marker),
        "channels_before_marker": all_channels_before_marker,
        "n_bad_by_channel_outlier_marker": len(marker_bad_channels),
        "bad_by_channel_outlier_marker": marker_bad_channels,
        "n_bad_by_trial_outliers": len(channels_to_drop) if outlier_policy in ["drop", "drop_and_nan", "drop_and_impute"] else 0,
        "bad_by_trial_outliers": channels_to_drop if outlier_policy in ["drop", "drop_and_nan", "drop_and_impute"] else [],
        # channels after both drops = channels after marker - trial-outlier drops
        "n_channels_after_all_drops": len(good.ch_names) - (len(channels_to_drop) if outlier_policy in ["drop", "drop_and_nan", "drop_and_impute"] else 0),
    }

    qc_filepath = os.path.join(save_dir, f"{sub}_channel_qc_summary.json")
    print(f"DEBUG: qc_filepath = '{qc_filepath}'")
    with open(qc_filepath, "w") as f:
        json.dump(qc_summary, f, indent=4)

    print(f"Saved QC summary for {sub} to {qc_filepath}")

# <<< Step 2: PROCESS the baseline ONCE >>>
base_trials = trial_ieeg_rand_offset(good, baseline_event, within_base_times, base_times_length, pad_length, preload=True)
if outlier_policy == 'drop':
    base_trials.drop_channels(channels_to_drop, on_missing='ignore')
elif outlier_policy == 'nan':
    outliers_to_nan(base_trials, outliers=outliers)
elif outlier_policy == 'drop_and_nan':
    base_trials.drop_channels(channels_to_drop, on_missing='ignore')
    outliers_to_nan(base_trials, outliers=outliers)
elif outlier_policy == 'drop_and_impute':
    base_trials.drop_channels(channels_to_drop, on_missing='ignore')
    outliers_to_nan(base_trials, outliers=outliers)
    impute_trial_nans_by_channel_mean(base_trials)
else:
    print('ignoring outliers')
if filter_method == 'filterbank_hilbert':    
    HG_base = gamma.extract(base_trials, passband=passband, copy=False, n_jobs=1)
elif filter_method == 'bandpass':
    HG_base = base_trials.copy().filter(passband[0], passband[1], method=method, fir_design=fir_design)
    HG_base.apply_hilbert(envelope=True)
else:
    raise ValueError("Please choose filterbank_hilbert or bandpass as your filter method. Other filter methods are not yet supported.")
pad_length_string = f"{pad_length}s"
crop_pad(HG_base, pad_length_string)
HG_base.decimate(dec_factor)

# Square the data to get power from amplitude (FIXED: Now correctly placed)
HG_base_power = HG_base.copy()
HG_base_power._data = HG_base._data ** 2
    
if isinstance(stat_func, partial):
    base_func_name = stat_func.func.__name__
    # Create a descriptive name like "ttest_ind_equal_var_False"
    keywords_str = "_".join(f"{k}_{v}" for k, v in sorted(stat_func.keywords.items()))
    if keywords_str: # If there are keywords like equal_var
        stat_func_for_filename = f"{base_func_name}_{keywords_str}"
    else: # If partial was used without keywords (less likely here)
        stat_func_for_filename = base_func_name
elif hasattr(stat_func, '__name__'): # For regular functions
    stat_func_for_filename = stat_func.__name__
elif isinstance(stat_func, str): # If a string was somehow passed (e.g., from a less robust CLI)
    # Sanitize or use the string directly if it's simple.
    # For safety, you might want to ensure it's a valid filename component.
    stat_func_for_filename = stat_func.replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
else:
    stat_func_for_filename = "custom_stat_func" # Fallback
    
output_name_base = f"{base_times_length}sec_within{within_base_times[0]}-{within_base_times[1]}sec_base_decFactor_{dec_factor}_outliers_{outliers}_{outlier_policy}_thresh_perc_{threshold_percent}_{passband[0]}-{passband[1]}_Hz_padLength_{pad_length}s_stat_func_{stat_func_for_filename}"
for event in ["Stimulus", "Response"]:
    print(f"--- Processing Event: {event} ---")
    output_name_event = f'{event}_{times[0]}to{times[1]}sec_{output_name_base}'

    times_adj = [times[0] - pad_length, times[1] + pad_length]
    
    trials = trial_ieeg(good, event, times_adj, preload=True, reject_by_annotation=False)
    trials.metadata = make_metadata_from_event_names(trials) # add metadata so we can grab specific trial types later. Untested 2/5/26.
    trials.metadata = add_previous_trial_info(trials.metadata)

    if outlier_policy == 'drop':
        trials.drop_channels(channels_to_drop, on_missing='ignore')
    elif outlier_policy == 'nan':
        outliers_to_nan(trials, outliers=outliers)
    elif outlier_policy == 'drop_and_nan':
        trials.drop_channels(channels_to_drop, on_missing='ignore')
        outliers_to_nan(trials, outliers=outliers)
    elif outlier_policy == 'drop_and_impute':
        trials.drop_channels(channels_to_drop, on_missing='ignore')
        outliers_to_nan(trials, outliers=outliers)
        impute_trial_nans_by_channel_mean(trials)
    else:
        print('ignoring outliers')

    # After dropping channels, update the channels list to match the data
    channels = trials.ch_names
    
    # Now extract gamma and proceed with analysis
    if filter_method == 'filterbank_hilbert':
        HG_ev1 = gamma.extract(trials, passband=passband, copy=True, n_jobs=1)
    elif filter_method == 'bandpass':
        HG_ev1 = trials.copy().filter(passband[0], passband[1], method=method, fir_design=fir_design)
        HG_ev1.apply_hilbert(envelope=True) # get real signal only - if you set envelope=False, you can grab the complex signal and then do np.angle(HG_ev1._data) to get the phase. But make sure to do that before decimation cuz that will mess up the phase calculation.
    else:
        raise ValueError("Please choose filterbank_hilbert or bandpass as your filter method. Other filter methods are not yet supported.")
    
    crop_pad(HG_ev1, pad_length_string)
    HG_ev1.decimate(dec_factor)
    # Square the data to get power from amplitude
    HG_ev1_power = HG_ev1.copy()
    HG_ev1_power._data = HG_ev1._data ** 2 # Square amplitude to get power

    # get the rescaled amplitude
    HG_ev1_rescaled = rescale(HG_ev1, HG_base, copy=True, mode='zscore')

    # get the rescaled power
    HG_ev1_power_rescaled = rescale(HG_ev1_power, HG_base_power, copy=True, mode='zscore')

    # get the evoke and evoke rescaled amplitude
    HG_ev1_evoke = HG_ev1.average(method=lambda x: np.nanmean(x, axis=0)) #axis=0 should be set for actually running this, the axis=2 is just for drift testing.
    HG_ev1_evoke_rescaled = HG_ev1_rescaled.average(method=lambda x: np.nanmean(x, axis=0))

    # get the evoke and evoke power rescaled amplitude
    HG_ev1_evoke_power = HG_ev1_power.average(method=lambda x: np.nanmean(x, axis=0)) #axis=0 should be set for actually running this, the axis=2 is just for drift testing.
    HG_ev1_evoke_power_rescaled = HG_ev1_power_rescaled.average(method=lambda x: np.nanmean(x, axis=0))

    # Save HG_ev1
    HG_ev1.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1-epo.fif', overwrite=True)
    HG_ev1.metadata.to_csv(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_metadata.csv', index=False)
    
    HG_ev1_power.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_power-epo.fif', overwrite=True)
    HG_ev1_power.metadata.to_csv(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_power_metadata.csv', index=False)

    # Save HG_base (the shuffled version)
    HG_base.save(f'{save_dir}/{sub}_{output_name_event}_HG_base-epo.fif', overwrite=True)
    HG_base_power.save(f'{save_dir}/{sub}_{output_name_event}_HG_base_power-epo.fif', overwrite=True)

    # Save HG_ev1_rescaled
    HG_ev1_rescaled.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_rescaled-epo.fif', overwrite=True)
    HG_ev1_rescaled.metadata.to_csv(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_rescaled_metadata.csv', index=False)

    HG_ev1_power_rescaled.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_power_rescaled-epo.fif', overwrite=True)
    HG_ev1_power_rescaled.metadata.to_csv(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_power_rescaled_metadata.csv', index=False)

    # Save HG_ev1_evoke
    HG_ev1_evoke.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_evoke-ave.fif', overwrite=True)
    HG_ev1_evoke_power.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_evoke_power-ave.fif', overwrite=True)

    # Save HG_ev1_evoke_rescaled
    HG_ev1_evoke_rescaled.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_evoke_rescaled-ave.fif', overwrite=True)
    HG_ev1_evoke_power_rescaled.save(f'{save_dir}/{sub}_{output_name_event}_HG_ev1_evoke_power_rescaled-ave.fif', overwrite=True)
    
    ###
    print(f"Shape of HG_ev1._data: {HG_ev1._data.shape}")
    print(f"Shape of HG_base._data: {HG_base._data.shape}")
    
    # oh this changed and returns both the significant clusters matrix and the p values now
    mat, _ = time_perm_cluster(HG_ev1._data, HG_base._data, 0.05, n_jobs=6, ignore_adjacency=1, stat_func=stat_func)

    #save channels with their indices 
    save_channels_to_file(channels, sub, task, save_dir)

    # save significant channels to a json
    save_sig_chans(f'{output_name_event}', mat, channels, sub, save_dir)
    
    # Assuming `mat` is your array and `save_dir` is the directory where you want to save it
    mat_save_path = os.path.join(save_dir, f'{output_name_event}_mat.npy')

    # Save the mat array
    np.save(mat_save_path, mat)
    
    # save dropped channels
    dropped_channels_filepath = os.path.join(save_dir, f'{sub}_{output_name_event}_dropped_channels.json')
    with open(dropped_channels_filepath, 'w') as f:
        json.dump({'dropped_channels': channels_to_drop}, f, indent=4)
    print(f"saved list of dropped channels to: {dropped_channels_filepath}")


NameError: name 'sub' is not defined